In [7]:
import os
os.environ["DISABLE_TRANSFORMERS_FLASH_ATTN"] = "1"

## Load dataset

In [8]:
import pandas as pd
df = df = pd.read_csv("quality_data.csv")
print(df.head())

                      narration        mode    type    category   subcategory
0        emi auto debit sbi acc  AUTO_DEBIT   Debit     expense          loan
1      salary credited via hdfc        NEFT  Credit      income        salary
2             upi txn to zomato         UPI   Debit     expense          food
3  paid electricity bill online  NETBANKING   Debit     expense         bills
4      mutual fund sip deducted  AUTO_DEBIT   Debit  investment  mutual_funds


# Create input_text

In [9]:
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

## Lowercase

In [10]:
df["input_text"] = df["input_text"].str.lower()

# Create label (Ground Truth)

In [11]:
df["label"] = df["category"] + "_" + df["subcategory"]

In [12]:
print(df[["input_text", "label"]].head())

                                          input_text                    label
0  narration: emi auto debit sbi acc | mode: auto...             expense_loan
1  narration: salary credited via hdfc | mode: ne...            income_salary
2  narration: upi txn to zomato | mode: upi | typ...             expense_food
3  narration: paid electricity bill online | mode...            expense_bills
4  narration: mutual fund sip deducted | mode: au...  investment_mutual_funds


## Prepare Dataset for HuggingFace

In [13]:
from datasets import Dataset

df["input_text"] = (
    df["narration"] + " " + df["mode"] + " " + df["type"]
)

df["input_text"] = df["input_text"].str.lower()

dataset = Dataset.from_pandas(df[["input_text", "label"]])

## Encode Labels to IDs

In [14]:
labels = list(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

def encode(example):
    example["label"] = label2id[example["label"]]
    return example

dataset = dataset.map(encode)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

## Tokenization

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["input_text"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

## Train-Test Split

In [16]:
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

## Load Model + Training Setup

In [17]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=6,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.6) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/240 [00:00<?, ?it/s]

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': 2.6091, 'grad_norm': 5.6848344802856445, 'learning_rate': 4.791666666666667e-05, 'epoch': 0.25}
{'loss': 2.4006, 'grad_norm': 7.3854217529296875, 'learning_rate': 4.5833333333333334e-05, 'epoch': 0.5}
{'loss': 2.2699, 'grad_norm': 7.10662317276001, 'learning_rate': 4.375e-05, 'epoch': 0.75}
{'loss': 2.0785, 'grad_norm': 9.305754661560059, 'learning_rate': 4.166666666666667e-05, 'epoch': 1.0}
{'loss': 1.7108, 'grad_norm': 6.012136936187744, 'learning_rate': 3.958333333333333e-05, 'epoch': 1.25}
{'loss': 1.6968, 'grad_norm': 6.925281047821045, 'learning_rate': 3.7500000000000003e-05, 'epoch': 1.5}
{'loss': 1.463, 'grad_norm': 7.388950347900391, 'learning_rate': 3.541666666666667e-05, 'epoch': 1.75}
{'loss': 1.3233, 'grad_norm': 7.675487518310547, 'learning_rate': 3.3333333333333335e-05, 'epoch': 2.0}
{'loss': 1.171, 'grad_norm': 5.740159034729004, 'learning_rate': 3.125e-05, 'epoch': 2.25}
{'loss': 0.8536, 'grad_norm': 6.456779956817627, 'learning_rate': 2.916666666666667e-05, '

TrainOutput(global_step=240, training_loss=1.0216116706530254, metrics={'train_runtime': 1543.8202, 'train_samples_per_second': 0.614, 'train_steps_per_second': 0.155, 'total_flos': 125608207749120.0, 'train_loss': 1.0216116706530254, 'epoch': 6.0})

## Trainer Setup + Model Training

In [18]:
def explain_distilbert(text, tokenizer, model, labels, top_n=3):
    import torch

    inputs = tokenizer(text, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    probs = torch.softmax(logits, dim=1)[0].tolist()

    # Top prediction
    top_idx = probs.index(max(probs))
    top_label = labels[top_idx]
    top_prob = probs[top_idx]

    # Second best
    sorted_indices = sorted(range(len(probs)), key=lambda i: probs[i])
    second_idx = sorted_indices[-2]
    second_label = labels[second_idx]
    second_prob = probs[second_idx]

    # Tokens
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    tokens = [t.replace("##", "") for t in tokens if t not in ["[CLS]", "[SEP]", "[PAD]"]]
    tokens = [t for t in tokens if len(t) > 2][:top_n]

    return f"Tokens [{', '.join(tokens)}] → '{top_label}' ({top_prob:.2f}), next '{second_label}' ({second_prob:.2f})"

## Explainability Function

In [19]:
import torch
import torch.nn.functional as F

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)

    confidence = torch.max(probs).item()
    pred_id = torch.argmax(probs).item()

    label = id2label[pred_id]
    category, subcategory = label.split("_", 1)

    return category, subcategory, round(confidence, 2)

def confidence_level(conf):
    if conf >= 0.65:
        return "High"
    elif conf >= 0.4:
        return "Medium"
    else:
        return "Low"

## Prediction Function

In [20]:
results = []

for i in range(len(df)):
    narration = df["narration"].iloc[i]
    mode = df["mode"].iloc[i]
    txn_type = df["type"].iloc[i]
    text = df["input_text"].iloc[i]

    pred_category, pred_subcategory, confidence = predict(text)

    model_comment = explain_distilbert(text, tokenizer, model, labels)

    results.append({
        "narration": narration,
        "mode": mode,
        "type": txn_type,
        "category": df["category"].iloc[i],
        "subcategory": df["subcategory"].iloc[i],
        "predicted_category": pred_category,
        "predicted_subcategory": pred_subcategory,
        "confidence": confidence,
        "confidence_percent": f"{round(confidence*100,1)}%",
        "confidence_level": confidence_level(confidence),
        "correct": (df["category"].iloc[i] == pred_category) and 
                   (df["subcategory"].iloc[i] == pred_subcategory),

        "model_comment": model_comment
    })

## Confidence Level Classification

In [21]:
result_df = pd.DataFrame(results)

In [22]:
result_df.head(20)

,narration,mode,type,category,subcategory,predicted_category,predicted_subcategory,confidence,confidence_percent,confidence_level,correct,model_comment
0,emi auto debit sbi acc,AUTO_DEBIT,Debit,expense,loan,expense,loan,0.90,90.0%,High,True,"Tokens [emi, auto, bit] → 'expense_loan' (0.92..."
1,salary credited via hdfc,NEFT,Credit,income,salary,income,salary,0.92,92.0%,High,True,"Tokens [salary, credited, via] → 'income_salar..."
2,upi txn to zomato,UPI,Debit,expense,food,expense,food,0.81,81.0%,High,True,"Tokens [oma, bit] → 'expense_food' (0.83), nex..."
3,paid electricity bill online,NETBANKING,Debit,expense,bills,expense,bills,0.89,89.0%,High,True,"Tokens [paid, electricity, bill] → 'expense_bi..."
4,mutual fund sip deducted,AUTO_DEBIT,Debit,investment,mutual_funds,expense,loan,0.81,81.0%,High,False,"Tokens [mutual, fund, sip] → 'expense_loan' (0..."
5,got interest from fd,NEFT,Credit,investment,fd_interest,investment,fd_interest,0.53,53.0%,Medium,True,"Tokens [got, interest, from] → 'investment_fd_..."
6,amazon pay txn,UPI,Debit,expense,shopping,expense,shopping,0.75,75.0%,High,True,"Tokens [amazon, pay, bit] → 'expense_shopping'..."
7,credited bonus amount,NEFT,Credit,income,salary,income,salary,0.91,91.0%,High,True,"Tokens [credited, bonus, amount] → 'income_sal..."
8,fuel payment hp petrol,UPI,Debit,expense,transport,expense,transport,0.91,91.0%,High,True,"Tokens [fuel, payment, petrol] → 'expense_tran..."
9,insurance premium paid,AUTO_DEBIT,Debit,expense,insurance,expense,insurance,0.85,85.0%,High,True,"Tokens [insurance, premium, paid] → 'expense_i..."


## | Metric | Meaning      | Use here               |
### | ------ | ------------ | ---------------------- |
### | F1     | balance      | good baseline          |
### | F2     | recall heavy | better for your system |


In [23]:
y_true = result_df["category"] + "_" + result_df["subcategory"]
y_pred = result_df["predicted_category"] + "_" + result_df["predicted_subcategory"]

In [24]:
from sklearn.metrics import f1_score, fbeta_score

f1 = f1_score(y_true, y_pred, average='weighted')
f2 = fbeta_score(y_true, y_pred, beta=2, average='weighted')

print("Overall Metrics:")
print(f"F1 Score: {round(f1,3)}")
print(f"F2 Score: {round(f2,3)}")

Overall Metrics:
F1 Score: 0.933
F2 Score: 0.943


## Classification Report

In [25]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_true, y_pred, output_dict=True)

report_df = pd.DataFrame(report).transpose()

print("\nClassification Report:")
print(report_df)


Classification Report:
                         precision    recall  f1-score     support
expense_bills             0.954545  1.000000  0.976744   21.000000
expense_food              1.000000  1.000000  1.000000   20.000000
expense_health            1.000000  1.000000  1.000000    7.000000
expense_insurance         1.000000  0.928571  0.962963   14.000000
expense_loan              0.869565  1.000000  0.930233   20.000000
expense_others            0.000000  0.000000  0.000000    2.000000
expense_shopping          0.950000  1.000000  0.974359   19.000000
expense_transport         1.000000  0.947368  0.972973   19.000000
income_others             0.875000  0.875000  0.875000    8.000000
income_salary             0.923077  1.000000  0.960000   24.000000
investment_fd_booking     1.000000  1.000000  1.000000    9.000000
investment_fd_interest    1.000000  1.000000  1.000000   16.000000
investment_mutual_funds   0.000000  0.000000  0.000000    4.000000
investment_others         0.000000  0.

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

## F2 Score per Class

In [26]:
from sklearn.metrics import fbeta_score

labels = list(set(y_true))

f2_scores = {}

for label in labels:
    y_true_bin = [1 if y == label else 0 for y in y_true]
    y_pred_bin = [1 if y == label else 0 for y in y_pred]

    f2_scores[label] = fbeta_score(y_true_bin, y_pred_bin, beta=2)

report_df["f2_score"] = report_df.index.map(f2_scores)

In [27]:
report_df = report_df.round(3)
print("\nClassification Report:")
print(report_df)


Classification Report:
                         precision  recall  f1-score  support  f2_score
expense_bills                0.955   1.000     0.977   21.000     0.991
expense_food                 1.000   1.000     1.000   20.000     1.000
expense_health               1.000   1.000     1.000    7.000     1.000
expense_insurance            1.000   0.929     0.963   14.000     0.942
expense_loan                 0.870   1.000     0.930   20.000     0.971
expense_others               0.000   0.000     0.000    2.000     0.000
expense_shopping             0.950   1.000     0.974   19.000     0.990
expense_transport            1.000   0.947     0.973   19.000     0.957
income_others                0.875   0.875     0.875    8.000     0.875
income_salary                0.923   1.000     0.960   24.000     0.984
investment_fd_booking        1.000   1.000     1.000    9.000     1.000
investment_fd_interest       1.000   1.000     1.000   16.000     1.000
investment_mutual_funds      0.000   0.0

## Overall Metrics

In [28]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred)

print("\nOverall Metrics:")
print("Accuracy:", round(accuracy, 3))
print("Macro F1:", report_df.loc["macro avg", "f1-score"])
print("Weighted F1:", report_df.loc["weighted avg", "f1-score"])


Overall Metrics:
Accuracy: 0.949
Macro F1: 0.772
Weighted F1: 0.933


## Best & Worst Performing Categories

In [29]:
print("\nBest Performing Category:")
print(report_df["f1-score"].idxmax(), report_df["f1-score"].max())

print("\nWorst Performing Category:")
print(report_df["f1-score"].idxmin(), report_df["f1-score"].min())


Best Performing Category:
expense_food 1.0

Worst Performing Category:
expense_others 0.0


In [30]:
model.save_pretrained("distilbert_model")
tokenizer.save_pretrained("distilbert_model")

('distilbert_model\\tokenizer_config.json',
 'distilbert_model\\special_tokens_map.json',
 'distilbert_model\\vocab.txt',
 'distilbert_model\\added_tokens.json',
 'distilbert_model\\tokenizer.json')

In [31]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("distilbert_model")
model = AutoModelForSequenceClassification.from_pretrained("distilbert_model")

In [32]:
model.save_pretrained("distilbert_model")
tokenizer.save_pretrained("distilbert_model")

('distilbert_model\\tokenizer_config.json',
 'distilbert_model\\special_tokens_map.json',
 'distilbert_model\\vocab.txt',
 'distilbert_model\\added_tokens.json',
 'distilbert_model\\tokenizer.json')

In [33]:
import joblib

joblib.dump(label2id, "label2id.pkl")
joblib.dump(id2label, "id2label.pkl")

['id2label.pkl']

In [34]:
trainer.train()

  0%|          | 0/240 [00:00<?, ?it/s]

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': 0.1808, 'grad_norm': 1.2273876667022705, 'learning_rate': 4.791666666666667e-05, 'epoch': 0.25}
{'loss': 0.1704, 'grad_norm': 1.5422170162200928, 'learning_rate': 4.5833333333333334e-05, 'epoch': 0.5}
{'loss': 0.3383, 'grad_norm': 1.991681694984436, 'learning_rate': 4.375e-05, 'epoch': 0.75}
{'loss': 0.1585, 'grad_norm': 2.7791807651519775, 'learning_rate': 4.166666666666667e-05, 'epoch': 1.0}
{'loss': 0.1396, 'grad_norm': 0.37995198369026184, 'learning_rate': 3.958333333333333e-05, 'epoch': 1.25}
{'loss': 0.1282, 'grad_norm': 5.411797523498535, 'learning_rate': 3.7500000000000003e-05, 'epoch': 1.5}
{'loss': 0.1219, 'grad_norm': 0.2581517696380615, 'learning_rate': 3.541666666666667e-05, 'epoch': 1.75}
{'loss': 0.0508, 'grad_norm': 0.9808741807937622, 'learning_rate': 3.3333333333333335e-05, 'epoch': 2.0}
{'loss': 0.1102, 'grad_norm': 0.5275931358337402, 'learning_rate': 3.125e-05, 'epoch': 2.25}
{'loss': 0.0478, 'grad_norm': 0.42742210626602173, 'learning_rate': 2.91666666666

TrainOutput(global_step=240, training_loss=0.08182856999337673, metrics={'train_runtime': 1498.5793, 'train_samples_per_second': 0.633, 'train_steps_per_second': 0.16, 'total_flos': 125608207749120.0, 'train_loss': 0.08182856999337673, 'epoch': 6.0})

In [35]:
model.save_pretrained("distilbert_model")
tokenizer.save_pretrained("distilbert_model")

('distilbert_model\\tokenizer_config.json',
 'distilbert_model\\special_tokens_map.json',
 'distilbert_model\\vocab.txt',
 'distilbert_model\\added_tokens.json',
 'distilbert_model\\tokenizer.json')

In [36]:
import joblib

label2id = model.config.label2id
id2label = model.config.id2label

joblib.dump(label2id, "label2id.pkl")
joblib.dump(id2label, "id2label.pkl")

['id2label.pkl']

In [37]:
print(model.config.id2label)

{0: 'expense_loan', 1: 'income_salary', 2: 'expense_food', 3: 'expense_bills', 4: 'investment_mutual_funds', 5: 'investment_fd_interest', 6: 'expense_shopping', 7: 'expense_transport', 8: 'expense_insurance', 9: 'investment_stocks', 10: 'expense_others', 11: 'income_others', 12: 'investment_others', 13: 'investment_fd_booking', 14: 'expense_health'}
